In [1]:
!pip install jiwer
!pip install evaluate
!pip install pytorch_lightning
!pip install unbabel-comet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
import re
import json
import random
import logging
import evaluate
import torch
import torchaudio
import pytorch_lightning as pl

from tqdm import tqdm
from pathlib import Path
from datasets import load_dataset
from dataclasses import dataclass
from pytorch_lightning import Trainer
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [3]:
from google.colab import drive

drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device: cuda
GPU: Tesla T4
memory: 15.6 GB


In [4]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [5]:
AUDIO_DIR = Path("/content/drive/MyDrive/KPI/audio/Lab-3")

In [6]:
dataset = load_dataset(
    "nvidia/Granary",
    "uk",
    split="ast"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

uk/ytc/uk_asr.jsonl:   0%|          | 0.00/2.82M [00:00<?, ?B/s]

uk/ytc/uk_ast-en.jsonl:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

Generating asr split: 0 examples [00:00, ? examples/s]

Generating ast split: 0 examples [00:00, ? examples/s]

In [7]:
def split_data(items):
    items = list(items)
    random.shuffle(items)

    test_size = int(0.1 * len(items))
    val_size = int(0.1 * len(items))

    test = items[:test_size]
    val = items[test_size:test_size + val_size]
    train = items[test_size + val_size:]

    return train, val, test

In [8]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\sа-яіїєґ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [9]:
class GranaryDataset(Dataset):
    def __init__(self, items, processor):
        self.items = [
            x for x in items
            if os.path.exists(AUDIO_DIR / x["audio_filepath"])
        ]
        self.processor = processor

    @staticmethod
    def safe_load_audio(path):
        try:
            audio, sr = torchaudio.load(path)
            if audio is None or audio.numel() == 0:
                return None
            return audio, sr
        except Exception:
            return None

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        audio_data = self.safe_load_audio(AUDIO_DIR / item["audio_filepath"])

        if audio_data is None:
            return self.__getitem__((idx + 1) % len(self.items))

        audio, sr = audio_data
        audio = audio.mean(dim=0)

        if sr != 16000:
            audio = torchaudio.functional.resample(audio, sr, 16000)

        target_text = normalize_text(item["answer"])

        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        labels = self.processor.tokenizer(
            target_text,
            return_tensors="pt"
        ).input_ids

        return {
            "input_features": inputs.input_features[0],
            "labels": labels[0]
        }

In [10]:
@dataclass
class DataCollator:
    processor: any

    def __call__(self, batch):
        input_features = [b["input_features"] for b in batch]
        labels = [b["labels"] for b in batch]

        input_features = torch.nn.utils.rnn.pad_sequence(
            input_features, batch_first=True
        )

        attention_mask = torch.ones_like(input_features[:, :, 0])

        labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )

        return {
            "input_features": input_features,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [11]:
class WhisperLightning(pl.LightningModule):
    def __init__(self, model_name, processor, lr=1e-5):
        super().__init__()
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)

        self.model.generation_config.suppress_tokens = []
        self.model.generation_config.forced_decoder_ids = None

        self.processor = processor
        self.lr = lr

    def training_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("train_loss", out.loss, prog_bar=True, on_step=True, on_epoch=True)
        return out.loss

    def validation_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("val_loss", out.loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

In [12]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny",
    language="uk",
    task="translate",
)

items = dataset

train_items, val_items, test_items = split_data(items)

train_ds = GranaryDataset(train_items, processor)
val_ds = GranaryDataset(val_items, processor)

collator = DataCollator(processor)

train_loader = DataLoader(
    train_ds, batch_size=8, num_workers=4, shuffle=True, collate_fn=collator,
)
val_loader = DataLoader(
    val_ds, batch_size=8, num_workers=4, collate_fn=collator,
)

model = WhisperLightning.load_from_checkpoint(
    AUDIO_DIR / "whisper_lightning_trans.ckpt",
    model_name="openai/whisper-tiny",
    processor=processor
)

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [13]:
from comet import download_model, load_from_checkpoint

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def evaluate_model(model, dataloader, processor):
    preds, refs = [], []

    model.eval()
    for batch in tqdm(dataloader):
        with torch.no_grad():
            pred_ids = model.model.generate(
                batch["input_features"].to(model.device),
                language="uk",
                task="translate",
            )

        pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        ref_str = processor.batch_decode(labels, skip_special_tokens=True)

        preds.extend(pred_str)
        refs.extend(ref_str)

    return preds, refs

def save_comet_triplets(src, ref, hyp, save_dir="./comet_eval"):
    os.makedirs(save_dir, exist_ok=True)
    src_path = os.path.join(save_dir, "src.txt")
    ref_path = os.path.join(save_dir, "ref.txt")
    hyp_path = os.path.join(save_dir, "hyp.txt")

    with open(src_path, "w", encoding="utf-8") as f:
        f.write("\n".join(src) + "\n")
    with open(ref_path, "w", encoding="utf-8") as f:
        f.write("\n".join(ref) + "\n")
    with open(hyp_path, "w", encoding="utf-8") as f:
        f.write("\n".join(hyp) + "\n")

    return src_path, ref_path, hyp_path

def compute_comet(src, ref, hyp, model_name="Unbabel/wmt22-comet-da", batch_size=8):
    comet_path = download_model(model_name)
    comet_model = load_from_checkpoint(comet_path)

    comet_data = [
        {"src": s, "mt": h, "ref": r}
        for s, r, h in zip(src, ref, hyp)
    ]

    comet_output = comet_model.predict(
        comet_data,
        batch_size=batch_size,
        gpus=1 if torch.cuda.is_available() else 0,
    )
    return comet_output.system_score

In [16]:
next(model.model.parameters()).device

device(type='cuda', index=0)

In [15]:
model.to("cuda")

WhisperLightning(
  (model): WhisperForConditionalGeneration(
    (model): WhisperModel(
      (encoder): WhisperEncoder(
        (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 384)
        (layers): ModuleList(
          (0-3): 4 x WhisperEncoderLayer(
            (self_attn): WhisperAttention(
              (k_proj): Linear(in_features=384, out_features=384, bias=False)
              (v_proj): Linear(in_features=384, out_features=384, bias=True)
              (q_proj): Linear(in_features=384, out_features=384, bias=True)
              (out_proj): Linear(in_features=384, out_features=384, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (fc2): Linea

In [17]:
test_ds = GranaryDataset(test_items, processor)
test_loader = DataLoader(
    test_ds, batch_size=8, num_workers=4, collate_fn=collator,
)

preds, refs = evaluate_model(model, test_loader, processor)
srcs = [item["text"] for item in test_ds.items]

src_path, ref_path, hyp_path = save_comet_triplets(srcs, refs, preds)
print(f"Saved COMET files:\n- src: {src_path}\n- ref: {ref_path}\n- hyp: {hyp_path}")

indices = random.sample(range(len(preds)), min(10, len(preds)))
print("Sample predictions:")
for idx in indices:
    print(f"Src: {srcs[idx]}")
    print(f"Ref: {refs[idx]}")
    print(f"Pred: {preds[idx]}")
    print("-" * 20)

wer = wer_metric.compute(predictions=preds, references=refs)
cer = cer_metric.compute(predictions=preds, references=refs)
print(f"WER: {wer}, CER: {cer}")

comet_score = compute_comet(srcs, refs, preds)
print(f"COMET: {comet_score:.4f}")

  0%|          | 0/36 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 36/36 [01:04<00:00,  1.79s/it]


Saved COMET files:
- src: ./comet_eval/src.txt
- ref: ./comet_eval/ref.txt
- hyp: ./comet_eval/hyp.txt
Sample predictions:
Src: І варто приїжджати, щоби підкорювати небо. Вітаю, Ігоре! Раді вас бачити. До неба підвезете? Без проблем. Клас. Ну, полетіли? Поїдемо. Такий потужний апарат. Як він називається? Це називається паратрайк. В нього стоїть двигун від справжнього самольоту. Він може полетіти дуже вирішно.
Ref: and its worth coming to conquer the sky congratulations igor were glad to see you will you take us to the sky no problem great well are we flying lets go such a powerful machine whats it called its called a paratroyk it has an engine from a real airplane it can fly very high
Pred:  and its worth coming to conquer the sky congratulations igor were glad to see you are welcome no problem great well are we flying lets go such a powerful machine which is called a paratroyk it has a different kind of a different kind of people it can fly very high
--------------------
Src: Ну, це 5

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages

COMET: 0.6515


In [ ]:
preds

In [ ]:
refs